In [15]:
import gym
import numpy as np

In [16]:
env = gym.make("CartPole-v1")

#set hyperparameters
alpha = 0.001
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
min_epsilon = 0.01
episodes = 100

#initialize Q-table
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))
 

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
#build the q network
def build_model(state_size, action_size):


    # Build the Q-network
    model = Sequential()
    model.add(Dense(24, input_dim=state_size, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(action_size, activation='linear'))

    model.compile(loss='mse', optimizer=Adam(learning_rate=alpha))
    return model

In [ ]:
#build the Q-network
q_network = build_model(state_size, action_size)
for _ in range(episodes):
    state, info = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0
    for time in range(500):
        if np.random.rand() <= epsilon:
            action = np.random.choice(action_size)
        else:
            q_values = q_network.predict(state)
            action = np.argmax(q_values[0])
        next_state, reward, done, truncated, _ = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])
        total_reward += reward
        if done: reward = -10
        q_target = q_network.predict(state)
        q_target[0][action] = reward + gamma * np.max(q_network.predict(next_state)[0])
        q_network.fit(state, q_target, epochs=1, verbose=0)
        state = next_state
        if done:
            print(f"Episode: {_ + 1}/{episodes}, score: {total_reward}")
            break
    if epsilon > min_epsilon:
        epsilon *= epsilon_decay